# Deepfake (CIFAKE) detector — training v2

Changes vs v1 (which got 71% test accuracy):
1. **Upsample 32→224 inside the model.** EfficientNetB0 downsamples by 32×, so a 32×32 input collapsed to a single 1×1 feature cell — the ImageNet features never saw a usable image. The model *input* stays 32×32, so the backend needs no change.
2. **Fine-tune the whole backbone** (v1 trained only the last 20 layers at lr 1e-5 and was still underfitting: train acc < val acc).
3. **Augmentation = horizontal flip only.** Rotation/zoom on 32×32 images re-interpolates pixels and blurs the generator artifacts we're trying to detect.
4. **Proper per-epoch shuffling**, cosine LR schedule, checkpoint on best val AUC, more epochs.
5. **Dataset sanity check** — v1 found 200k train / 30k test images, but CIFAKE is 100k / 20k (balanced). Check this before training.

In [1]:
import os, json, hashlib
from pathlib import Path
from collections import Counter
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers

print("TF:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

# walk up from wherever the notebook runs until we find the project folder (the one containing backend/)
ROOT = Path.cwd()
while not (ROOT / "backend" / "train").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

DATA_DIR  = ROOT / "backend" / "train" / "dataset"
MODEL_DIR = ROOT / "backend" / "saved_models"
print("Project:", ROOT)
print("Data   :", DATA_DIR)
assert (DATA_DIR / "train").exists() and (DATA_DIR / "test").exists(), f"dataset not found at {DATA_DIR}"

TF: 2.16.2
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Project: /Users/priyankareddy/Downloads/ai-deepfakedetection-main
Data   : /Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/train/dataset


## 1. Dataset sanity check
Official CIFAKE: **train = 50,000 REAL + 50,000 FAKE, test = 10,000 REAL + 10,000 FAKE.**
v1 reported 200,000 train and 30,000 test (20k REAL vs 10k FAKE). That points to a duplicated or extra folder (e.g. the zip extracted twice into a nested folder, or a stray `REAL` copy). Fix the folders until the counts match, and make sure no test image also appears in train.

In [2]:
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def files(d):
    return [p for p in Path(d).rglob("*") if p.suffix.lower() in IMG_EXT]

for split in ["train", "test"]:
    for cls in sorted(os.listdir(DATA_DIR / split)):
        if (DATA_DIR / split / cls).is_dir():
            print(f"{split:5s} {cls:5s} {len(files(DATA_DIR / split / cls)):>7,d}")

def md5(p):
    return hashlib.md5(p.read_bytes()).hexdigest()

train_hashes = Counter(md5(p) for p in files(DATA_DIR / "train"))
test_hashes  = Counter(md5(p) for p in files(DATA_DIR / "test"))
print("duplicate files inside train:", sum(c - 1 for c in train_hashes.values()))
print("duplicate files inside test :", sum(c - 1 for c in test_hashes.values()))
print("test images also in train   :", len(set(train_hashes) & set(test_hashes)))

train fake  100,000
train real  100,000
test  fake   10,000
test  real   20,000
duplicate files inside train: 920
duplicate files inside test : 0
test images also in train   : 0


## 2. Data pipeline
Images are kept at native 32×32 (cheap to cache in RAM); upsampling happens on the GPU inside the model.
`UP_SIZE = 224` gives the best accuracy. If an epoch is too slow on your machine, use 128 (about 3× faster, typically ~1 point lower).

In [3]:
IMG_SIZE = 32      # native CIFAKE resolution; the backend also resizes uploads to this
UP_SIZE  = 224     # what EfficientNet actually sees (try 128 if too slow)
BATCH    = 64
SEED     = 42
AUTOTUNE = tf.data.AUTOTUNE

def load(split, **kw):
    return tf.keras.utils.image_dataset_from_directory(
        str(DATA_DIR / split), image_size=(IMG_SIZE, IMG_SIZE),
        batch_size=None, label_mode="binary", **kw)

train_raw = load("train", validation_split=0.1, subset="training",   seed=SEED)
val_raw   = load("train", validation_split=0.1, subset="validation", seed=SEED)
test_raw  = load("test", shuffle=False)
print(train_raw.class_names)        # ['fake', 'real'] -> 0 = FAKE, 1 = REAL

# cache THEN shuffle, so every epoch sees a new order (v1 cached a fixed order)
train_ds = (train_raw.cache()
            .shuffle(20_000, seed=SEED, reshuffle_each_iteration=True)
            .batch(BATCH).prefetch(AUTOTUNE))
val_ds  = val_raw.cache().batch(BATCH).prefetch(AUTOTUNE)
test_ds = test_raw.batch(BATCH).prefetch(AUTOTUNE)

Found 200000 files belonging to 2 classes.
Using 180000 files for training.


2026-11-14 11:47:10.566822: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4
2026-11-14 11:47:10.566865: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 16.00 GB
2026-11-14 11:47:10.566870: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 5.92 GB
2026-11-14 11:47:10.566906: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-11-14 11:47:10.566919: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 200000 files belonging to 2 classes.
Using 20000 files for validation.
Found 30000 files belonging to 2 classes.
['fake', 'real']


## 3. Model
Input 32×32 (0–255) → flip → resize to UP_SIZE → EfficientNetB0 (has its own normalisation, so keep 0–255) → GAP → sigmoid P(REAL).

In [4]:
base = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_shape=(UP_SIZE, UP_SIZE, 3))
base.trainable = False

inputs  = layers.Input((IMG_SIZE, IMG_SIZE, 3))
x = layers.RandomFlip("horizontal")(inputs)
x = layers.Resizing(UP_SIZE, UP_SIZE, interpolation="bilinear")(x)
x = base(x, training=False)          # BatchNorm stays in inference mode
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)   # P(REAL)

model = tf.keras.Model(inputs, outputs)
METRICS = ["accuracy", tf.keras.metrics.AUC(name="auc")]
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)      │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ random_flip (RandomFlip)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ efficientnetb0 (Functional)     │ (None, 7, 7, 1280)     │     4,049,571 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │         1,281 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,050,852 (15.45 MB)

 Trainable params: 1,281 (5.00 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

## 4. Phase 1 — train the head only (backbone frozen)

In [5]:
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="binary_crossentropy", metrics=METRICS)
history1 = model.fit(train_ds, validation_data=val_ds, epochs=2)

Epoch 1/2


/Users/priyankareddy/Downloads/ai-deepfakedetection-main/backend/.venv/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-11-14 11:47:14.309518: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


2813/2813 ━━━━━━━━━━━━━━━━━━━━ 1391s 491ms/step - accuracy: 0.7652 - auc: 0.8521 - loss: 0.4756 - val_accuracy: 0.8100 - val_auc: 0.8992 - val_loss: 0.4075
Epoch 2/2
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 1481s 526ms/step - accuracy: 0.7840 - auc: 0.8716 - loss: 0.4449 - val_accuracy: 0.8114 - val_auc: 0.9021 - val_loss: 0.4042


## 5. Phase 2 — fine-tune the whole backbone
All conv layers unfrozen (BatchNorm frozen, the standard recipe for small batches). Cosine LR decay from 2e-4, so the pretrained features are adjusted without being wiped out. The best epoch by val AUC is saved straight into `saved_models/model.keras`.

In [6]:
base.trainable = True
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

EPOCHS = 10
steps  = int(train_ds.cardinality()) if int(train_ds.cardinality()) > 0 else 2813
lr = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=2e-4, decay_steps=steps * EPOCHS, alpha=0.01)

model.compile(optimizer=tf.keras.optimizers.AdamW(lr, weight_decay=1e-4),
              loss="binary_crossentropy", metrics=METRICS)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(str(MODEL_DIR / "model.keras"),
        monitor="val_auc", mode="max", save_best_only=True),
    tf.keras.callbacks.EarlyStopping(monitor="val_auc", mode="max",
        patience=3, restore_best_weights=True),
]
history2 = model.fit(train_ds, validation_data=val_ds,
                     epochs=EPOCHS, callbacks=callbacks)

Epoch 1/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 5805s 2s/step - accuracy: 0.8681 - auc: 0.9422 - loss: 0.2948 - val_accuracy: 0.9035 - val_auc: 0.9705 - val_loss: 0.2223
Epoch 2/10
2813/2813 ━━━━━━━━━━━━━━━━━━━━ 4922s 2s/step - accuracy: 0.8981 - auc: 0.9645 - loss: 0.2411 - val_accuracy: 0.9043 - val_auc: 0.9660 - val_loss: 0.2396
Epoch 3/10
   1/2813 ━━━━━━━━━━━━━━━━━━━━ 1:26:30 2s/step - accuracy: 0.9062 - auc: 0.9835 - loss: 0.1987

KeyboardInterrupt: 

In [7]:
model = tf.keras.models.load_model(MODEL_DIR / "model.keras", compile=False)

## 6. Test-set evaluation

In [8]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, roc_auc_score

y_true = np.concatenate([y.numpy() for _, y in test_ds]).ravel().astype(int)
p_real = model.predict(test_ds, verbose=0).ravel()
# test-time augmentation: average with the horizontally flipped image
p_flip = model.predict(test_ds.map(lambda x, y: (tf.image.flip_left_right(x), y)), verbose=0).ravel()
p_tta  = (p_real + p_flip) / 2

for name, p in [("plain", p_real), ("flip-TTA", p_tta)]:
    y_pred = (p > 0.5).astype(int)
    print(f"{name:8s} acc={np.mean(y_pred == y_true):.4f}  "
          f"balanced_acc={balanced_accuracy_score(y_true, y_pred):.4f}  "
          f"AUC={roc_auc_score(y_true, p):.4f}")

y_pred = (p_real > 0.5).astype(int)
print(classification_report(y_true, y_pred, target_names=["FAKE", "REAL"], digits=4))
print(confusion_matrix(y_true, y_pred))
test_acc = float(np.mean(y_pred == y_true))

2026-11-14 15:34:32.627411: W tensorflow/core/framework/local_rendezvous.cc:404] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


plain    acc=0.8872  balanced_acc=0.8895  AUC=0.9590
flip-TTA acc=0.8920  balanced_acc=0.8947  AUC=0.9620
              precision    recall  f1-score   support

        FAKE     0.7923    0.8966    0.8412     10000
        REAL     0.9447    0.8824    0.9125     20000

    accuracy                         0.8872     30000
   macro avg     0.8685    0.8895    0.8769     30000
weighted avg     0.8939    0.8872    0.8887     30000

[[ 8966  1034]
 [ 2351 17649]]


## 7. Save (format unchanged, so the FastAPI backend works as-is)

In [9]:
model.save(MODEL_DIR / "model.keras")
meta = {
    "model": "EfficientNetB0",
    "input_size": [IMG_SIZE, IMG_SIZE],
    "backbone_input_size": [UP_SIZE, UP_SIZE],
    "labels": {"0": "FAKE", "1": "REAL"},
    "preprocessing": "RGB, resize to 32x32, keep values 0-255 (no rescaling); model upsamples internally",
    "output": "sigmoid, P(REAL)",
    "test_accuracy": test_acc,
}
with open(MODEL_DIR / "meta.json", "w") as f:
    json.dump(meta, f, indent=2)
print(json.dumps(meta, indent=2))

{
  "model": "EfficientNetB0",
  "input_size": [
    32,
    32
  ],
  "backbone_input_size": [
    224,
    224
  ],
  "labels": {
    "0": "FAKE",
    "1": "REAL"
  },
  "preprocessing": "RGB, resize to 32x32, keep values 0-255 (no rescaling); model upsamples internally",
  "output": "sigmoid, P(REAL)",
  "test_accuracy": 0.8871666666666667
}
